In [49]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

data = np.load("../../FluidGPT_validation/tensordata/amira__0__AR-d2__.npz", allow_pickle=True)
data2 = np.load("../../FluidGPT_validation/tensordata/amira__0__AR-d3__.npz", allow_pickle=True)
data3 = np.load("../../FluidGPT_validation/tensordata/amira__0__FM-d2__.npz", allow_pickle=True)
data4 = np.load("../../FluidGPT_validation/tensordata/amira__0__FM-d3__.npz", allow_pickle=True)

datasetlist = ['amira'] #, 'pdebench-comp', 'pdebench-incomp', 'pdegym-gauss', 'pdegym-sines', 'pdegym-bb', 'pdegym-pwc', 'pdegym-svs', 'pdegym-sl']
trajlist = [0] #,1,2,3,4,5,6,7,8,9]
models = ['AR-d2', 'AR-d3', 'FM-d2', 'FM-d3']
base_dir = "../../FluidGPT_validation/tensordata/"




# See what arrays are stored
print(data.files)
ground_truth = data['traj_pred']
prediction = data['traj_true']
print(ground_truth.shape, prediction.shape)
print(ground_truth.dtype, prediction.dtype)

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 150  # value in MB, default is 20

['label', 'model_type', 'dsplit_idx', 'traj_idx', 'actual_traj_idx', 'traj_pred', 'traj_true', 'rae_errors', 'rrmse_errors', 'cfg']
(1, 201, 2, 128, 128) (1, 201, 2, 128, 128)
float32 float32


## DUE TO A BUG THE GROUND TRUTH AND PREDICTIONS ARE REVERSED!!!

In [54]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
QUANTITY = 'radial'   # 'vorticity' or 'radial'
CMAP = 'viridis'
CMAP_ERROR = 'magma'
FPS_INTERVAL = 80

# ------------------------------------------------------------------
# FIELD COMPUTATION
# ------------------------------------------------------------------
def compute_vorticity(traj):
    vx, vy = traj[:, 0], traj[:, 1]
    dvy_dx = np.gradient(vy, axis=2)
    dvx_dy = np.gradient(vx, axis=1)
    return dvy_dx - dvx_dy

def compute_speed(traj):
    vx, vy = traj[:, 0], traj[:, 1]
    return np.sqrt(vx**2 + vy**2)

def compute_field(traj, quantity):
    if quantity == 'vorticity':
        return compute_vorticity(traj)
    elif quantity == 'radial':
        return compute_speed(traj)
    else:
        raise ValueError("quantity must be 'vorticity' or 'radial'")

for dataset in datasetlist:
    for traj in trajlist:
        raw_data = []
        for model in models:
            file_dir = f"{base_dir}{dataset}__{traj}__{model}__.npz"
            raw_data.append(np.load(file_dir, allow_pickle=True))
        data = raw_data[0]  # Use the first model's data as reference
        ground_truth = data['traj_pred'].squeeze(0)
        print(ground_truth.shape, ground_truth.dtype)
        data2 = raw_data[1]
        data3 = raw_data[2]
        data4 = raw_data[3]

        datasets = {
            'AR-d2': data,
            'AR-d3': data2,
            'FM-d2': data3,
            'FM-d3': data4,
            }

        field_gt = compute_field(ground_truth, QUANTITY)
        field_pred = {k: compute_field(v['traj_true'][0], QUANTITY) for k, v in datasets.items()}
        field_err = {k: np.abs(field_pred[k] - field_gt) for k in datasets}  # absolute error

        n_frames = field_gt.shape[0]
        print(f"Number of frames: {n_frames}")
        quantity_label = 'Vorticity' if QUANTITY == 'vorticity' else 'Velocity Magnitude'

        # Color scales
        all_main = [field_gt] + list(field_pred.values())
        if QUANTITY == 'vorticity':
            vmax = max(np.abs(f).max() for f in all_main)
            vmin = -vmax
        else:
            vmin = 0.0
            vmax = max(f.max() for f in all_main)

        #err_vmin = 0.0
        #err_vmax = max(f.max() for f in field_err.values())
        err_vmin = vmin
        err_vmax = vmax

        # ------------------------------------------------------------------
        # DARK-BACKGROUND / TRANSPARENT STYLING
        # ------------------------------------------------------------------
        TEXT_COLOR = 'white'

        def style_axis(ax, emphasize=False):
            ax.set_facecolor('none')
            ax.patch.set_alpha(0.0)
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_color(TEXT_COLOR)
                spine.set_alpha(0.9 if emphasize else 0.4)
                spine.set_linewidth(1.5 if emphasize else 1.0)

        # ------------------------------------------------------------------
        # LAYOUT: error (outer) | prediction (inner) | GT (center) | prediction (inner) | error (outer)
        # ------------------------------------------------------------------
        fig = plt.figure(figsize=(16, 8))
        fig.patch.set_alpha(0.0)

        gs = gridspec.GridSpec(
            2, 5, figure=fig,
            width_ratios=[0.6, 1, 1.3, 1, 0.6],
            wspace=0.05, hspace=0.15,
            top=0.90, bottom=0.16, left=0.02, right=0.98
        )

        # label -> (row, side)  side: 'L' = left half, 'R' = right half
        layout = {
            'AR-d2': (0, 'L'),   # top-left inner
            'FM-d2': (1, 'L'),   # bottom-left inner
            'AR-d3': (0, 'R'),   # top-right inner
            'FM-d3': (1, 'R'),   # bottom-right inner
        }

        pred_col = {'L': 1, 'R': 3}   # inner columns (next to GT)
        err_col  = {'L': 0, 'R': 4}   # outer columns

        pred_axes, pred_images = {}, {}
        err_axes, err_images = {}, {}

        for label, (r, side) in layout.items():
            # Prediction panel (inner)
            ax_p = fig.add_subplot(gs[r, pred_col[side]])
            im_p = ax_p.imshow(field_pred[label][0], cmap=CMAP, vmin=vmin, vmax=vmax, origin='lower')
            ax_p.set_title(label, fontsize=11, color=TEXT_COLOR, fontweight='bold')
            style_axis(ax_p)
            pred_axes[label] = ax_p
            pred_images[label] = im_p

            # Error panel (outer) — created in its gridspec slot, then shrunk + pulled inward
            ax_e = fig.add_subplot(gs[r, err_col[side]])
            im_e = ax_e.imshow(field_err[label][0], cmap=CMAP_ERROR, vmin=err_vmin, vmax=err_vmax, origin='lower')
            ax_e.set_title(f"{label} |error|", fontsize=8, color=TEXT_COLOR, alpha=0.85)
            style_axis(ax_e)

            # --- shrink and shift toward center ---
            pos = ax_e.get_position()  # Bbox(x0, y0, x1, y1) in figure fraction coords
            shrink = 1               # scale factor: smaller = smaller panel
            shift_frac = 0.1          # how far to pull inward, as fraction of panel width

            new_width = pos.width * shrink
            new_height = pos.height * shrink
            # center vertically within the original slot
            new_y0 = pos.y0 + (pos.height - new_height) / 2

            if side == 'L':
                # left-side error panel: shift right (toward center)
                new_x0 = pos.x0 + pos.width * shift_frac
            else:
                # right-side error panel: shift left (toward center)
                new_x0 = pos.x1 - new_width - pos.width * shift_frac

            ax_e.set_position([new_x0, new_y0, new_width, new_height])
            # ---------------------------------------

            err_axes[label] = ax_e
            err_images[label] = im_e

        # Ground truth (center, spans both rows)
        ax_gt = fig.add_subplot(gs[:, 2])
        im_gt = ax_gt.imshow(field_gt[0], cmap=CMAP, vmin=vmin, vmax=vmax, origin='lower')
        ax_gt.set_title('Ground Truth', fontsize=13, color=TEXT_COLOR, fontweight='bold')
        style_axis(ax_gt, emphasize=True)

        # ------------------------------------------------------------------
        # COLORBARS: horizontal, bottom of figure, under their respective blocks
        # ------------------------------------------------------------------
        cbar_width = 0.35
        cbar_height = 0.025
        cbar_y = 0.06
        gap = 0.06

        # center the pair of colorbars in the figure
        total_span = 2 * cbar_width + gap
        left_start = (1 - total_span) / 2   # = 0.12 with these numbers

        # Main field colorbar (left)
        cbar_ax_main = fig.add_axes([left_start, cbar_y, cbar_width, cbar_height])
        cbar_ax_main.patch.set_alpha(0.0)
        cbar_main = fig.colorbar(im_gt, cax=cbar_ax_main, orientation='horizontal', label=quantity_label)
        cbar_main.ax.xaxis.label.set_color(TEXT_COLOR)
        cbar_main.ax.tick_params(colors=TEXT_COLOR)
        cbar_main.outline.set_edgecolor(TEXT_COLOR)

        # Error colorbar (right), same width/height, offset by width + gap
        cbar_ax_err = fig.add_axes([left_start + cbar_width + gap, cbar_y, cbar_width, cbar_height])
        cbar_ax_err.patch.set_alpha(0.0)
        cbar_err = fig.colorbar(err_images['FM-d3'], cax=cbar_ax_err, orientation='horizontal', label='|error|')
        cbar_err.ax.xaxis.label.set_color(TEXT_COLOR)
        cbar_err.ax.tick_params(colors=TEXT_COLOR, labelsize=8)
        cbar_err.outline.set_edgecolor(TEXT_COLOR)

        suptitle = fig.suptitle(f"{quantity_label} at t = 0", fontsize=16, color=TEXT_COLOR)

        # ------------------------------------------------------------------
        # ANIMATION
        # ------------------------------------------------------------------
        def update(frame):
            for label in layout:
                pred_images[label].set_data(field_pred[label][frame])
                err_images[label].set_data(field_err[label][frame])
            im_gt.set_data(field_gt[frame])
            suptitle.set_text(f"{quantity_label} at t = {frame}")
            return list(pred_images.values()) + list(err_images.values()) + [im_gt, suptitle]

        anim = FuncAnimation(fig, update, frames=n_frames, interval=FPS_INTERVAL, blit=False)
        plt.close(fig)

        HTML(anim.to_jshtml())

        fig.patch.set_facecolor('#12161b')
        fig.patch.set_alpha(1.0)
        anim.save(f'../../FluidGPT_validation/animations/anim_{quantity_label}_{dataset}_{traj}.mp4', writer='ffmpeg', fps=15, dpi=150)
        print(f"Saved animation for dataset '{dataset}', trajectory {traj} as 'anim_{quantity_label}_{dataset}_{traj}.mp4'")

(201, 2, 128, 128) float32
Number of frames: 201
Saved animation for dataset 'amira', trajectory 0 as 'anim_Velocity Magnitude_amira_0.mp4'
